# Market Basket Analysis Data Exploration
Demonstrating Apriori algorithm from mlxtend.

In [ ]:
import pandas as pd
import os
import kagglehub
from mlxtend.frequent_patterns import apriori, association_rules
import plotly.express as px

# Download latest version of the dataset
path = kagglehub.dataset_download("aslanahmedov/market-basket-analysis")
print("Path to dataset files:", path)

# Look for the csv file in the downloaded path
csv_file = None
for file in os.listdir(path):
    if file.endswith('.csv'):
        csv_file = os.path.join(path, file)
        break

if csv_file:
    df = pd.read_csv(csv_file, sep=';')
else:
    raise FileNotFoundError("CSV file not found in downloaded dataset")

# Note: The actual dataset column structure might differ from ['Transaction', 'Item'] 
# Assuming it has a transaction ID and item name, update 'Transaction' and 'Item' accordingly:
# Replace 'Transaction' and 'Item' below with the actual names from df.columns if different.
transaction_col = df.columns[0]
item_col = df.columns[1]

# One-hot encode transactions
basket = (df.groupby([transaction_col, item_col])[item_col]
          .count().unstack().reset_index().fillna(0)
          .set_index(transaction_col))
basket_sets = basket.applymap(lambda x: 1 if x > 0 else 0)

# Apriori algorithm
frequent_itemsets = apriori(basket_sets, min_support=0.01, use_colnames=True)
rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1.0)

# Scatter plot logic
fig = px.scatter(rules, x="confidence", y="lift", size="support", color="lift", hover_data=["antecedents", "consequents"])
fig.show()

rules.head()